# 🚀 Optimized Mask2Former (4-Channel, 1024x1024, Memory Efficient)
Includes Dice Loss, Gradient Checkpointing, FP16, and Tiling Strategy.
Uses same structure and assumptions as your previous code.

In [1]:
!pip install -q transformers accelerate rasterio evaluate datasets albumentations opencv-python-headless
!pip install -q git+https://github.com/huggingface/transformers.git


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0")

import glob
import json
import rasterio
import numpy as np
import cv2
import torch
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation
from torch.cuda.amp import autocast, GradScaler

# =========================================
# GPU OPTIMIZATION
# =========================================
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# =========================================
# Dataset Path (DGX only)
# =========================================
DATASET_PATH = "/scratch/cs25m121/Data/FinalGeoData"

ANN_DIR = os.path.join(DATASET_PATH, "annotations")
IMAGE_DIR = os.path.join(DATASET_PATH, "images")
MASK_DIR = os.path.join(DATASET_PATH, "masks")

missing_dirs = [p for p in [ANN_DIR, IMAGE_DIR, MASK_DIR] if not os.path.isdir(p)]
if missing_dirs:
    raise FileNotFoundError(
        f"ERROR: Missing required DGX dataset folders: {missing_dirs}"
    )

# =========================================
# DGX Output Paths
# =========================================
RUN_ROOT = os.path.join(DATASET_PATH, "runs")
CHECKPOINT_DIR = os.path.join(RUN_ROOT, "mask2former-fresh-run")
BEST_MODEL_DIR = os.path.join(DATASET_PATH, "mask2former-best-fresh")
FINAL_MODEL_DIR = os.path.join(DATASET_PATH, "mask2former-final-fresh")

for out_dir in [RUN_ROOT, CHECKPOINT_DIR, BEST_MODEL_DIR, FINAL_MODEL_DIR]:
    os.makedirs(out_dir, exist_ok=True)

print(f"Dataset root: {DATASET_PATH}")
print(f"Annotations: {ANN_DIR}")
print(f"Images: {IMAGE_DIR}")
print(f"Masks: {MASK_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Best model dir: {BEST_MODEL_DIR}")
print(f"Final model dir: {FINAL_MODEL_DIR}")

# reduce for stability first, scale later
SUBSET_LIMIT = 500

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [24]:
all_ann_files = sorted(glob.glob(os.path.join(ANN_DIR, '*.json')))

# 🔥 limit scan for speed (no impact on training later)
scan_files = all_ann_files[:2000]

unique_classes = set()

for ann_path in tqdm(scan_files, desc="Scanning Types"):
    with open(ann_path, 'r') as f:
        data = json.load(f)

    for obj in data.get('objects', []):
        base_class = obj['class_name']
        attrs = obj.get('attributes', {})

        # 🔥 fast + clean mapping
        if base_class == "road":
            fine_type = attrs.get('road_type', "unknown")
        elif base_class == "builtup":
            fine_type = attrs.get('roof_type', "unknown")
        elif base_class == "water":
            fine_type = attrs.get('water_type', "unknown")
        else:
            fine_type = "base"

        # 🔥 normalize labels (IMPORTANT for IoU)
        fine_type = str(fine_type).lower().strip()

        class_str = f"{base_class}_type_{fine_type}"
        unique_classes.add(class_str)

# 🔥 deterministic + stable ordering
unique_classes = sorted(unique_classes)

id2label = {0: "background"}
for i, label in enumerate(unique_classes):
    id2label[i + 1] = label

label2id = {v: k for k, v in id2label.items()}

print(f"✅ Total classes: {len(id2label)}")

In [ ]:
class FinalGeoDataset(Dataset):
    def __init__(self, ann_files, processor, label2id, tile_size=1024):
        self.ann_files = ann_files
        self.processor = processor
        self.label2id = label2id
        self.tile_size = tile_size

    def __len__(self):
        return len(self.ann_files)

    def _load_sample(self, ann_path):
        basename = os.path.basename(ann_path).replace('.json', '.tif')
        image_path = os.path.join(IMAGE_DIR, basename)
        mask_path = os.path.join(MASK_DIR, basename)

        if os.path.exists(image_path):
            tif_path = image_path
        elif os.path.exists(mask_path):
            tif_path = mask_path
        else:
            return None

        with rasterio.open(tif_path) as src:
            image_array = src.read()
            image_array = np.transpose(image_array, (1, 2, 0))
            transform = src.transform
            inv_transform = ~transform

        h, w, c = image_array.shape
        if c < 4:
            pad = np.zeros((h, w, 4 - c), dtype=image_array.dtype)
            image_array = np.concatenate([image_array, pad], axis=2)
        elif c > 4:
            image_array = image_array[:, :, :4]

        H, W, C = image_array.shape
        tile_size = self.tile_size

        if H > tile_size and W > tile_size:
            x = np.random.randint(0, W - tile_size)
            y = np.random.randint(0, H - tile_size)
            image_array = image_array[y:y+tile_size, x:x+tile_size]
            transform = rasterio.Affine(
                transform.a, transform.b, transform.c + x * transform.a,
                transform.d, transform.e, transform.f + y * transform.e
            )
            inv_transform = ~transform
        else:
            image_array = image_array[:tile_size, :tile_size]

        with open(ann_path, 'r') as f:
            data = json.load(f)

        instance_map = np.zeros(image_array.shape[:2], dtype=np.int32)
        instance_id_to_semantic_id = {0: 0}

        for i, obj in enumerate(data.get('objects', [])):
            inst_id = i + 1
            geom = obj['geometry_geojson']

            base_class = obj['class_name']
            attrs = obj.get('attributes', {})

            if base_class == "road":
                fine_type = attrs.get('road_type', "unknown")
            elif base_class == "builtup":
                fine_type = attrs.get('roof_type', "unknown")
            elif base_class == "water":
                fine_type = attrs.get('water_type', "unknown")
            else:
                fine_type = "base"

            class_str = f"{base_class}_type_{str(fine_type).lower().strip()}"
            semantic_id = self.label2id.get(class_str, 0)

            if semantic_id == 0:
                continue

            if geom['type'] == 'Polygon':
                geo_coords = geom['coordinates'][0]
                pixel_coords = [inv_transform * pt for pt in geo_coords]
                pixel_coords = np.array(pixel_coords, np.int32)

                if pixel_coords.size == 0:
                    continue

                pixel_coords[:, 0] = np.clip(pixel_coords[:, 0], 0, tile_size - 1)
                pixel_coords[:, 1] = np.clip(pixel_coords[:, 1], 0, tile_size - 1)

                if len(np.unique(pixel_coords, axis=0)) < 3:
                    continue

                cv2.fillPoly(instance_map, [pixel_coords], inst_id)
                instance_id_to_semantic_id[inst_id] = semantic_id

        if len(instance_id_to_semantic_id) == 1 or not np.any(instance_map):
            return None

        image_array = image_array.astype("uint8")
        img_rgb = image_array[:, :, :3]
        extra_ch = image_array[:, :, 3:4]

        inputs = self.processor(
            images=img_rgb,
            segmentation_maps=instance_map,
            instance_id_to_semantic_id=instance_id_to_semantic_id,
            return_tensors="pt"
        )

        mask_labels = inputs.get("mask_labels", [])
        class_labels = inputs.get("class_labels", [])
        valid_mask_labels = []
        valid_class_labels = []
        for mask_label, class_label in zip(mask_labels, class_labels):
            mask_array = mask_label.detach().cpu().numpy() if torch.is_tensor(mask_label) else np.asarray(mask_label)
            if mask_array.size == 0 or not np.any(mask_array):
                continue
            valid_mask_labels.append(mask_label)
            valid_class_labels.append(class_label)

        if len(valid_mask_labels) == 0:
            return None

        inputs["mask_labels"] = valid_mask_labels
        inputs["class_labels"] = valid_class_labels

        pixel_values = inputs["pixel_values"].squeeze(0)
        extra_ch = torch.tensor(extra_ch).permute(2, 0, 1).float() / 255.0
        inputs["pixel_values"] = torch.cat([pixel_values, extra_ch], dim=0)

        processed_inputs = {}
        for k, v in inputs.items():
            if isinstance(v, list):
                processed_inputs[k] = v[0]
            else:
                processed_inputs[k] = v.squeeze(0) if getattr(v, "ndim", 0) > 0 else v

        return processed_inputs

    def __getitem__(self, idx):
        for offset in range(len(self.ann_files)):
            candidate_idx = (idx + offset) % len(self.ann_files)
            sample = self._load_sample(self.ann_files[candidate_idx])
            if sample is not None:
                return sample
        raise RuntimeError("No valid samples found in dataset.")

In [ ]:
import os
import glob

print("Scanning dataset folders...\n")

dataset_roots = [DATASET_PATH, "/scratch/cs25m121/Data/FinalGeoData"]
seen = set()

for root in dataset_roots:
    if not root or root in seen:
        continue
    seen.add(root)

    if not os.path.isdir(root):
        print(f"Skipping missing root: {root}")
        continue

    ann_candidates = sorted(glob.glob(os.path.join(root, "annotations", "*.json")))
    img_candidates = sorted(glob.glob(os.path.join(root, "images", "*.tif")))
    mask_candidates = sorted(glob.glob(os.path.join(root, "masks", "*.tif")))

    print(f"Dataset root: {root}")
    print(f"   JSON: {len(ann_candidates)}")
    print(f"   IMG : {len(img_candidates)}")
    print(f"   MASK: {len(mask_candidates)}")

    tif_pool = img_candidates if len(img_candidates) > 0 else mask_candidates
    if len(ann_candidates) == 0 or len(tif_pool) == 0:
        print("   No valid annotation-image mapping found.\n")
        continue

    tif_map = {os.path.basename(p): p for p in tif_pool}
    sample_ann = ann_candidates[0]
    basename = os.path.basename(sample_ann).replace(".json", ".tif")

    print("   Sample mapping:")
    print(f"      JSON: {sample_ann}")
    print(f"      TIF : {tif_map.get(basename, 'Not Found')}")
    print()

In [ ]:
from transformers import (
    Mask2FormerConfig,
    Mask2FormerForUniversalSegmentation,
    Mask2FormerImageProcessor
)
import random
import torch
import torch.nn as nn

MODEL_NAME = "facebook/mask2former-swin-small-coco-instance"

image_processor = Mask2FormerImageProcessor.from_pretrained(MODEL_NAME)

image_processor.do_resize = False
image_processor.do_rescale = True
image_processor.do_normalize = True

# Keep background as class 0; use a separate ignore index for void pixels.
image_processor.ignore_index = 255
image_processor.reduce_labels = False

random.seed(42)
all_ann_files_shuffled = all_ann_files.copy()
random.shuffle(all_ann_files_shuffled)

train_size = int(len(all_ann_files_shuffled) * 0.9)

train_dataset = FinalGeoDataset(
    all_ann_files_shuffled[:train_size],
    image_processor,
    label2id,
    tile_size=1024
)

val_dataset = FinalGeoDataset(
    all_ann_files_shuffled[train_size:],
    image_processor,
    label2id,
    tile_size=1024
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

config = Mask2FormerConfig.from_pretrained(
    MODEL_NAME,
    id2label=id2label,
    label2id=label2id
)

if hasattr(config, "backbone_config") and config.backbone_config is not None:
    config.backbone_config.num_channels = 4
if hasattr(config, "num_channels"):
    config.num_channels = 4

config.dice_weight = 5.0
config.class_weight = 3.0
config.mask_weight = 5.0

config.num_queries = 150
config.train_num_points = 12544
config.oversample_ratio = 3.0
config.importance_sample_ratio = 0.75



# from_pretrained with ignore_mismatched_sizes=True creates the 4-channel projection
# automatically, so we do not manually patch the conv here.
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True
)

if hasattr(model.config, "backbone_config") and model.config.backbone_config is not None:
    model.config.backbone_config.num_channels = 4
if hasattr(model.model.pixel_level_module.encoder, "config"):
    model.model.pixel_level_module.encoder.config.num_channels = 4

torch.backends.cudnn.benchmark = True

print("READY: 4-channel model + synced backbone config")

In [31]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import TrainingArguments, Trainer
import torch
import os
import glob
import re

# =========================================
# COLLATE (SAFE FOR 4CH)
# =========================================
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]).float(),
        "pixel_mask": torch.stack([x["pixel_mask"] for x in batch]),
        "mask_labels": [x["mask_labels"] for x in batch],
        "class_labels": [x["class_labels"] for x in batch],
    }

class ScalarLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model(**inputs)
        loss = outputs.loss
        if loss is None:
            raise ValueError("Model did not return a loss.")
        loss = loss.mean() if getattr(loss, "ndim", 0) > 0 else loss
        loss = loss.squeeze()
        return (loss, outputs) if return_outputs else loss

def find_latest_checkpoint(checkpoint_dir):
    if not os.path.isdir(checkpoint_dir):
        return None
    candidates = [
        p for p in glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))
        if os.path.isdir(p)
    ]
    if not candidates:
        return None

    def step_num(path):
        m = re.search(r"checkpoint-(\d+)$", os.path.basename(path))
        return int(m.group(1)) if m else -1

    candidates.sort(key=step_num)
    return candidates[-1]

# =========================================
# TRAINING ARGS (DGX + best checkpoint)
# =========================================
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    learning_rate=3e-5,
    num_train_epochs=9,

    lr_scheduler_type="cosine",
    warmup_steps=100,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,

    fp16=True,

    evaluation_strategy="steps",
    eval_steps=800,

    logging_strategy="steps",
    logging_steps=100,
    logging_dir=os.path.join(CHECKPOINT_DIR, "logs"),

    save_strategy="steps",
    save_steps=800,
    save_total_limit=2,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    eval_accumulation_steps=5,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    report_to="none"
 )

# =========================================
# TRAINER
# =========================================
trainer = ScalarLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
 )

# =========================================
# TRAIN FRESH + SAVE BEST MODEL
# =========================================
print("Training started...")
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

print("Starting fresh training run. Existing checkpoints in other folders will not be used.")
train_output = trainer.train()

best_ckpt = trainer.state.best_model_checkpoint
if best_ckpt is None:
    print("WARNING: No best checkpoint detected. Saving current model to BEST_MODEL_DIR.")
else:
    print(f"Best checkpoint detected: {best_ckpt}")

trainer.save_model(BEST_MODEL_DIR)
image_processor.save_pretrained(BEST_MODEL_DIR)
print(f"Best model exported to: {BEST_MODEL_DIR}")

if best_ckpt and os.path.isdir(best_ckpt):
    print(f"Best checkpoint files kept at: {best_ckpt}")

In [ ]:
import os

save_path = FINAL_MODEL_DIR

os.makedirs(save_path, exist_ok=True)

# save final model + processor
trainer.save_model(save_path)
image_processor.save_pretrained(save_path)

print(f"Final model saved at: {save_path}")
print(f"Best model saved at: {BEST_MODEL_DIR}")